In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [4]:
# Cell 1: Install dependencies
!pip install -q tiktoken datasets tqdm

In [5]:
# Cell 2: Data Preparation
import os
import numpy as np
import tiktoken
from datasets import load_dataset
from tqdm import tqdm

# Load a small subset of TinyStories
dataset = load_dataset("roneneldan/TinyStories", split='train[:5000]') # 5000 stories for speed
split_dataset = dataset.train_test_split(test_size=0.1)

enc = tiktoken.get_encoding("gpt2")

def process(example):
    ids = enc.encode_ordinary(example['text']) 
    ids.append(enc.eot_token) # append End of Text token
    return {'ids': ids, 'len': len(ids)}

# Tokenize
tokenized = split_dataset.map(process, remove_columns=['text'], num_proc=4)

# Save to binary files
for split, dset in tokenized.items():
    arr_len = np.sum(dset['len'])
    filename = f'{split}.bin'
    arr = np.memmap(filename, dtype=np.uint16, mode='w+', shape=(arr_len,))
    
    idx = 0
    for example in tqdm(dset['ids'], desc=f'writing {filename}'):
        arr[idx : idx + len(example)] = example
        idx += len(example)
    arr.flush()

print("Data preparation complete! train.bin and test.bin created.")

Map (num_proc=4):   0%|          | 0/4500 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/500 [00:00<?, ? examples/s]

writing test.bin: 100%|██████████| 500/500 [00:00<00:00, 9443.99it/s]

Data preparation complete! train.bin and test.bin created.


In [6]:
from datasets import load_dataset

# Load Alpaca - the gold standard for chat fine-tuning
chat_dataset = load_dataset("tatsu-lab/alpaca", split='train[:5000]') # small slice for speed

def format_chat(example):
    # Combine instruction and output into a chat format
    text = f"<|user|>: {example['instruction']}\n<|assistant|>: {example['output']}"
    # Tokenize
    tokens = enc.encode_ordinary(text)
    tokens.append(enc.eot_token)
    return {'ids': tokens}

tokenized_chat = chat_dataset.map(format_chat, remove_columns=chat_dataset.column_names)
# Save this to 'chat.bin' just like you did for TinyStories

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [7]:
from datasets import load_dataset
import numpy as np

# 1. Load Alpaca (Instruction dataset)
chat_dataset = load_dataset("tatsu-lab/alpaca", split='train[:10000]') 

# 2. Format into a Chat Template
def process_chat(example):
    # This template is the "Secret Sauce" for chatting
    text = f"<|user|>\n{example['instruction']}\n<|assistant|>\n{example['output']}<|endoftext|>"
    ids = enc.encode_ordinary(text)
    return {'ids': ids, 'len': len(ids)}

tokenized_chat = chat_dataset.map(process_chat, remove_columns=chat_dataset.column_names)

# 3. Save to chat.bin
arr_len = np.sum(tokenized_chat['len'])
filename = 'chat.bin'
arr = np.memmap(filename, dtype=np.uint16, mode='w+', shape=(arr_len,))
idx = 0
for example in tokenized_chat['ids']:
    arr[idx : idx + len(example)] = example
    idx += len(example)
arr.flush()
print("Chat data ready!")

Chat data ready!


In [9]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# --- Hyperparameters ---
batch_size = 16 
block_size = 256 
max_iters = 5000      # Increased: bigger models need more steps to "wake up"
eval_interval = 200
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_embd = 768 
n_head = 12 
n_layer = 12 
dropout = 0.1
vocab_size = 50257 

# --- Architectural Components ---

class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        assert n_embd % n_head == 0
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.n_head = n_head
        self.n_embd = n_embd

    def forward(self, x):
        B, T, C = x.size() 
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)

        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        y = F.scaled_dot_product_attention(
            q, k, v, # Fixed: was (q, q, v) in your snippet
            attn_mask=None, 
            dropout_p=dropout if self.training else 0, 
            is_causal=True
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C) 
        y = self.resid_dropout(self.c_proj(y))
        return y

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd, bias=False),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd, bias=False),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention()
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = FeedForward(n_embd)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class NanoGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(vocab_size, n_embd),
            wpe = nn.Embedding(block_size, n_embd),
            drop = nn.Dropout(dropout),
            h = nn.ModuleList([Block() for _ in range(n_layer)]),
            ln_f = nn.LayerNorm(n_embd),
        ))
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight # Weight tying

        self.apply(self._init_weights)
        
        # Special scaled init for residual projections
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/torch.sqrt(torch.tensor(2 * n_layer)))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        B, T = idx.size()
        pos = torch.arange(0, T, dtype=torch.long, device=device)
        tok_emb = self.transformer.wte(idx) 
        pos_emb = self.transformer.wpe(pos) 
        
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            logits = self.lm_head(x[:, [-1], :]) 
            loss = None
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= block_size else idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [10]:
# Cell 4: Multi-GPU Training Engine (Memory Optimized)
import numpy as np
import torch
import torch.nn as nn

# 1. Load data
train_data_chat = np.memmap('chat.bin', dtype=np.uint16, mode='r')
train_data_stories = np.memmap('train.bin', dtype=np.uint16, mode='r')
val_data = train_data_chat 

# 2. Memory Management Settings
# If batch_size in Cell 3 is 16, accumulation_steps=4 makes the effective batch 64.
accumulation_steps = 4 

def get_batch(split):
    if split == 'train':
        use_chat = np.random.random() < 0.7
        data = train_data_chat if use_chat else train_data_stories
    else:
        data = val_data
        
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+block_size+1]).astype(np.int64)) for i in ix])
    return x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)

# 3. Initialize Model
model = NanoGPT()

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)

model.to(device)

# 4. Optimizer & Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_iters)

# 5. Training Loop with Gradient Accumulation
print(f"Starting training: 12 Layers, Effective Batch Size: {batch_size * accumulation_steps}")

for iter in range(max_iters):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    
    accumulated_loss = 0
    for _ in range(accumulation_steps):
        xb, yb = get_batch('train')
        
        # Use autocast if you want to speed up T4 training even more
        logits, loss = model(xb, yb)
        
        # Scale the loss by accumulation steps
        loss = loss.mean() / accumulation_steps
        loss.backward()
        accumulated_loss += loss.item()

    optimizer.step()
    scheduler.step()

    if iter % 100 == 0:
        lr_now = scheduler.get_last_lr()[0]
        # Multiply back to show the real loss value
        print(f"Iter {iter}, Loss: {accumulated_loss:.4f}, LR: {lr_now:.6f}")

# 6. Save correctly
if isinstance(model, nn.DataParallel):
    torch.save(model.module.state_dict(), 'nanogpt_v2.pt')
else:
    torch.save(model.state_dict(), 'nanogpt_v2.pt')

print("Training complete. Smarter weights saved as 'nanogpt_v2.pt'")

Using 2 GPUs!
Starting training: 12 Layers, Effective Batch Size: 64


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Iter 0, Loss: 10.9739, LR: 0.000300
Iter 100, Loss: 4.7650, LR: 0.000300
Iter 200, Loss: 4.2142, LR: 0.000299
Iter 300, Loss: 3.8070, LR: 0.000297
Iter 400, Loss: 3.5199, LR: 0.000295
Iter 500, Loss: 3.2760, LR: 0.000293
Iter 600, Loss: 2.9473, LR: 0.000289
Iter 700, Loss: 2.9334, LR: 0.000286
Iter 800, Loss: 2.5218, LR: 0.000281
Iter 900, Loss: 2.5282, LR: 0.000277
Iter 1000, Loss: 2.3124, LR: 0.000271
Iter 1100, Loss: 2.2977, LR: 0.000266
Iter 1200, Loss: 1.7828, LR: 0.000259
Iter 1300, Loss: 1.7252, LR: 0.000253
Iter 1400, Loss: 2.0143, LR: 0.000246
Iter 1500, Loss: 1.6496, LR: 0.000238
Iter 1600, Loss: 1.4680, LR: 0.000230
Iter 1700, Loss: 1.1024, LR: 0.000222
Iter 1800, Loss: 1.5378, LR: 0.000214
Iter 1900, Loss: 1.4591, LR: 0.000205
Iter 2000, Loss: 0.7853, LR: 0.000196
Iter 2100, Loss: 1.0255, LR: 0.000187
Iter 2200, Loss: 0.9258, LR: 0.000178
Iter 2300, Loss: 0.8816, LR: 0.000169
Iter 2400, Loss: 0.8101, LR: 0.000159
Iter 2500, Loss: 0.7632, LR: 0.000150
Iter 2600, Loss: 0.4380

In [11]:
# Updated Chat Cell for 12-Layer Model
import torch

# Ensure model is in eval mode
model.eval()

def generate_response(prompt, max_new_tokens=150):
    # Format the prompt using the tags the model learned
    context = f"<|user|>\n{prompt}\n<|assistant|>\n"
    x = torch.tensor(enc.encode_ordinary(context), dtype=torch.long, device=device).unsqueeze(0)
    
    # Generate tokens
    # We use model.module if we're still using DataParallel from training
    curr_model = model.module if isinstance(model, torch.nn.DataParallel) else model
    y = curr_model.generate(x, max_new_tokens=max_new_tokens, temperature=0.8, top_k=50)
    
    # Decode and clean up
    full_text = enc.decode(y[0].tolist())
    # Extract only the assistant's new response
    response = full_text[len(context):].split("<|endoftext|>")[0]
    return response.strip()

print("NanoGPT v2 is ready! (Type 'quit' to stop)")
while True:
    user_msg = input("You: ")
    if user_msg.lower() == 'quit': break
    
    response = generate_response(user_msg)
    print(f"NanoGPT: {response}")

NanoGPT v2 is ready! (Type 'quit' to stop)


You:  Hi


NanoGPT: My name is John and I'm a college student. I'm passionate about tackling my skills and am excited to discuss different perspectives and how to use technology to make the most of technology. I believe my skills and knowledge will be an asset to the development and to your skills. 

I have in the [date] improvements that I am striving to grow in the industry, and am confident that I can take on the business level.

Thank you for your time and consideration.

Sincerely, 
[Your Name]


You:  quit
